In [15]:
import pandas as pd
import numpy as np
import warnings 
import holidays
import matplotlib.pyplot as plt
import koreanize_matplotlib
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings('ignore')

In [16]:
df2024 = pd.read_csv('../../Data/Zero/2024_data.csv')
df2025 = pd.read_csv('../../Data/Zero/2025_data.csv')

df = pd.concat([df2024,df2025],axis=0)

In [ ]:
# 1. 시작 대여소 기준으로만 필터링 (순수 대여량 예측 목적)
target_station = ['ST-1035', 'ST-454', 'ST-471']
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])
df['weekday'] = df['기준_날짜'].dt.dayofweek
df['day_type'] = np.where(df['weekday'] < 5, 0, 1)
df = df[(df['전체_이용_분'] >= 5) & (df['전체_이용_거리'] >= 5)]

df_filtered = df[df['시작_대여소_ID'].isin(target_station)].copy()
# 2. 대여소별, 시간별로 정렬 (시계열의 생명은 순서!)
df_filtered = df_filtered.sort_values(by=['시작_대여소_ID', '기준_날짜', '시간대'])

# 3. 그룹별로 변수 생성 (이게 핵심입니다)
# transform을 사용하면 인덱스가 유지되면서 그룹 내에서만 shift/rolling이 일어납니다.
df_filtered['lag1'] = df_filtered.groupby('시작_대여소_ID')['전체_건수'].shift(1)
df_filtered['lag24'] = df_filtered.groupby('시작_대여소_ID')['전체_건수'].shift(24)
df_filtered['rolling3'] = df_filtered.groupby('시작_대여소_ID')['전체_건수'].transform(lambda x: x.rolling(3).mean())

# 4. 결측치 제거 (shift로 인해 생긴 첫 행들 제거)
df_filtered = df_filtered.dropna()

# 5. 모델 학습 시 '시작_대여소_ID'도 피처에 포함 (Label Encoding 후)
# HGB는 대여소마다의 고유 패턴을 학습해야 합니다.

In [24]:
target = '전체_건수'
features = [
    '시간대',
    'day_type',
    '온도',
    '습도',
    '불쾌지수',
    '강수량',
    '적설량',
    'lag1',
    'lag24',
    'rolling3',
]

In [26]:
train = df_filtered[df_filtered['기준_날짜'].dt.year == 2024]
test  = df_filtered[df_filtered['기준_날짜'].dt.year == 2025]

X_train = train[features]
y_train = train[target]

X_test = test[features]
y_test = test[target]

from sklearn.ensemble import HistGradientBoostingRegressor

hgb = HistGradientBoostingRegressor(
    learning_rate=0.03,
    max_iter=600,
    max_depth=10,
    min_samples_leaf=10,
    random_state=42
)

hgb.fit(X_train, y_train)
pred_hgb = hgb.predict(X_test)

print("HGB MAE:", mean_absolute_error(y_test, pred_hgb))
print("HGB RMSE:", np.sqrt(mean_squared_error(y_test, pred_hgb)))
print("HGB R2:", r2_score(y_test, pred_hgb))

HGB MAE: 0.047264929821390154
HGB RMSE: 0.16197979420907344
HGB R2: 0.5275578074888977


In [25]:
df_filtered

,기준_날짜,시간대,집계_기준,시작_대여소_ID,종료_대여소_ID,전체_건수,전체_이용_분,전체_이용_거리,온도,습도,불쾌지수,강수량,적설량,weekday,day_type,lag1,lag24,rolling3
133,2024-01-01,13,출발시간,ST-1035,ST-2264,1.0,6300,17438.0,6.1,68,45.62352,0.1,0.0,0,0,1.0,1.0,1.0
134,2024-01-01,13,출발시간,ST-1035,ST-2778,1.0,840,1490.0,6.1,68,45.62352,0.1,0.0,0,0,1.0,1.0,1.0
167,2024-01-01,16,출발시간,ST-1035,ST-3169,1.0,900,1721.0,6.1,65,45.87135,0.0,0.0,0,0,1.0,1.0,1.0
169,2024-01-01,16,출발시간,ST-1035,ST-3169,1.0,900,1721.0,6.1,65,45.87135,0.0,0.0,0,0,1.0,1.0,1.0
188,2024-01-01,17,출발시간,ST-1035,ST-1035,1.0,3900,7209.0,4.3,78,41.94946,0.0,0.0,0,0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160084,2025-12-31,15,출발시간,ST-471,ST-119,1.0,2400,7667.0,-4.4,31,36.95264,0.0,0.0,2,0,1.0,1.0,1.0
160132,2025-12-31,18,출발시간,ST-471,ST-231,1.0,2280,6074.0,-6.6,42,32.20372,0.0,0.0,2,0,1.0,1.0,1.0
160138,2025-12-31,19,출발시간,ST-471,ST-231,1.0,2280,6074.0,-7.2,43,31.25396,0.0,0.0,2,0,1.0,1.0,1.0
160149,2025-12-31,20,출발시간,ST-471,ST-480,1.0,540,1460.0,-7.9,45,29.94655,0.0,0.0,2,0,1.0,1.0,1.0


In [27]:
df['전체_건수'].value_counts()

전체_건수
1.0    321461
2.0     14306
3.0       836
4.0       116
5.0        22
6.0         8
7.0         2
Name: count, dtype: int64

In [28]:
df['전체_건수'].describe()

count    336751.000000
mean          1.048897
std           0.234468
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           7.000000
Name: 전체_건수, dtype: float64